<a href="https://colab.research.google.com/github/Pramila003/DeteministicAndVoiceSemantic/blob/main/deterministicEngine_atlas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.3 MB/s eta 0:00:00


In [ ]:
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

class DeterministicDocSearch:
    def __init__(self, chunk_size_words: int = 80, overlap_words: int = 20, k1: float = 1.5, b: float = 0.75):
        self.chunk_size = chunk_size_words
        self.overlap = overlap_words
        self.k1 = k1
        self.b = b

        self.chunks = {}              # chunk_id -> raw chunk text
        self.chunk_lengths = {}       # chunk_id -> token count
        self.avg_chunk_len = 0.0
        self.inverted_index = defaultdict(dict)  # token -> {chunk_id: tf}
        self.idf = {}

    def _tokenize(self, text: str) -> list[str]:
        """Normalize and tokenize text into words."""
        return re.findall(r'\b[a-z0-9]+\b', text.lower())

    def extract_text_from_file(self, file_path: str) -> str:
        """Extract plain text from .txt or .pdf files."""
        path = Path(file_path)
        if not path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")

        if path.suffix.lower() == ".txt":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()

        elif path.suffix.lower() == ".pdf":
            try:
                from pypdf import PdfReader
                reader = PdfReader(str(path))
                text = ""
                for page_num, page in enumerate(reader.pages, start=1):
                    extracted = page.extract_text() or ""
                    text += f"\n[Page {page_num}]\n" + extracted
                return text
            except ImportError:
                raise ImportError("Please run `pip install pypdf` to read PDF files.")
        else:
            raise ValueError(f"Unsupported file type: {path.suffix}. Use .txt or .pdf")

    def _chunk_text(self, full_text: str) -> list[str]:
        """Split text into overlapping chunks so context isn't cut off."""
        words = full_text.split()
        chunks = []
        step = self.chunk_size - self.overlap
        if step <= 0:
            step = self.chunk_size

        for i in range(0, len(words), step):
            chunk = " ".join(words[i:i + self.chunk_size])
            if chunk.strip():
                chunks.append(chunk)
        return chunks

    def load_document(self, file_path: str):
        """Read file, break into chunks, and index with BM25."""
        raw_text = self.extract_text_from_file(file_path)
        raw_chunks = self._chunk_text(raw_text)

        total_tokens = 0
        num_chunks = len(raw_chunks)

        self.chunks.clear()
        self.chunk_lengths.clear()
        self.inverted_index.clear()
        self.idf.clear()

        for idx, chunk_text in enumerate(raw_chunks):
            chunk_id = f"chunk_{idx + 1}"
            tokens = self._tokenize(chunk_text)

            self.chunks[chunk_id] = chunk_text
            self.chunk_lengths[chunk_id] = len(tokens)
            total_tokens += len(tokens)

            counts = Counter(tokens)
            for token, count in counts.items():
                self.inverted_index[token][chunk_id] = count

        self.avg_chunk_len = total_tokens / num_chunks if num_chunks > 0 else 0

        # Compute BM25 Inverse Document Frequency (IDF)
        for token, postings in self.inverted_index.items():
            df = len(postings)
            self.idf[token] = math.log(1.0 + (num_chunks - df + 0.5) / (df + 0.5))

        print(f"Document indexed successfully: {num_chunks} chunks created.")

    def search(self, query: str, top_k: int = 3) -> list[dict]:
        """Search the document and return ranked snippets."""
        query_tokens = self._tokenize(query)
        if not query_tokens:
            return []

        scores = defaultdict(float)

        for token in query_tokens:
            if token not in self.inverted_index:
                continue

            idf = self.idf[token]
            postings = self.inverted_index[token]

            for chunk_id, tf in postings.items():
                chunk_len = self.chunk_lengths[chunk_id]
                numerator = tf * (self.k1 + 1.0)
                denominator = tf + self.k1 * (1.0 - self.b + self.b * (chunk_len / self.avg_chunk_len))
                scores[chunk_id] += idf * (numerator / denominator)

        ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)

        results = []
        for chunk_id, score in ranked[:top_k]:
            if score > 0.0:
                results.append({
                    "chunk_id": chunk_id,
                    "score": round(score, 2),
                    "text": self.chunks[chunk_id]
                })

        return results

In [ ]:
engine = DeterministicDocSearch(chunk_size_words=40, overlap_words=10)

# 1. User uploads and indexes document
engine.load_document("/content/sample_contract.txt")

# 2. User searches inside that uploaded document
user_query = "how many days notice for termination?"
matches = engine.search(user_query, top_k=2)

print("\n--- Search Results ---")
for i, match in enumerate(matches, 1):
    print(f"\nResult {i} (Match Score: {match['score']}):")
    print(f"Snippet: \"{match['text']}\"")

FileNotFoundError: File not found: /content/sample_contract.txt

In [ ]:
!pip install sentence-transformers pypdf numpy

In [ ]:
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

class LocalVectorDocSearch:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", chunk_size_words: int = 60, overlap_words: int = 15):
        self.chunk_size = chunk_size_words
        self.overlap = overlap_words

        # 1. Load local open-source embedding model (runs on local CPU/GPU, no API key)
        print(f"Loading local embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)

        self.chunks = []            # List of text chunks
        self.embeddings = None      # 2D NumPy array: shape (num_chunks, vector_dimension)

    def extract_text(self, file_path: str) -> str:
        """Extract raw text from a local .txt or .pdf file."""
        path = Path(file_path)
        if not path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")

        if path.suffix.lower() == ".txt":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()

        elif path.suffix.lower() == ".pdf":
            from pypdf import PdfReader
            reader = PdfReader(str(path))
            pages = [page.extract_text() or "" for page in reader.pages]
            return "\n".join(pages)

        raise ValueError(f"Unsupported format: {path.suffix}. Use .txt or .pdf")

    def _chunk_text(self, text: str) -> list[str]:
        """Split text into overlapping word chunks."""
        words = text.split()
        chunks = []
        step = self.chunk_size - self.overlap
        if step <= 0:
            step = self.chunk_size

        for i in range(0, len(words), step):
            chunk = " ".join(words[i : i + self.chunk_size])
            if chunk.strip():
                chunks.append(chunk)
        return chunks

    def load_and_index(self, file_path: str):
        """Read document, chunk it, and convert chunks into dense vector embeddings."""
        raw_text = self.extract_text(file_path)
        self.chunks = self._chunk_text(raw_text)

        if not self.chunks:
            print("Document has no text content.")
            return

        print(f"Creating vector embeddings for {len(self.chunks)} chunks...")
        # Converts list of strings -> dense vectors (384 dimensions each)
        # normalize_embeddings=True allows computing cosine similarity via simple dot product
        self.embeddings = self.model.encode(
            self.chunks,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        print("Vector index built successfully.")

    def search(self, query: str, top_k: int = 3) -> list[dict]:
        """Convert query to a vector and compute cosine similarity against document vectors."""
        if self.embeddings is None or len(self.chunks) == 0:
            return []

        # 1. Embed user query into the same 384-dimensional vector space
        query_vector = self.model.encode(
            query,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        # 2. Compute Cosine Similarity: Dot product of normalized vectors
        # Shape: (num_chunks,)
        similarity_scores = np.dot(self.embeddings, query_vector)

        # 3. Get top_k indices sorted descending
        top_indices = np.argsort(similarity_scores)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append({
                "chunk_id": int(idx + 1),
                "score": float(similarity_scores[idx]),
                "text": self.chunks[idx]
            })

        return results

In [ ]:
if __name__ == "__main__":
    searcher = LocalVectorDocSearch()

    # Ingest and embed the document locally
    searcher.load_and_index("/content/sample_contract.txt")

    # Search with terms that have ZERO keyword overlap:
    # "holiday allowances" -> matches "annual leave" and "Paid Time Off"
    query = "how many days notice for termination?"

    print(f"\nSearching for: '{query}'\n")
    results = searcher.search(query, top_k=2)

    for i, res in enumerate(results, 1):
        print(f"Rank {i} (Similarity: {res['score']:.4f})")
        print(f"Chunk #{res['chunk_id']}: {res['text']}\n")

In [ ]:
### SENTIMENT ANLAYSIS


In [ ]:

!pip install -q transformers torch

In [ ]:
# Cell 2: Load model and run interactive classification
from transformers import pipeline


classifier = pipeline(
    "text-classification",
    model="tabularisai/multilingual-sentiment-analysis"
)


while True:
    user_text = input("Enter a sentence to analyze (or type 'exit' to quit): ").strip()

    if not user_text:
        print("Please enter some text.\n")
        continue

    if user_text.lower() in ["exit", "quit", "q"]:
        print("Exiting.")
        break

    # Run inference
    result = classifier(user_text)[0]
    label = result["label"]
    confidence = result["score"] * 100

    print(f"Predicted Sentiment : {label}")
    print(f"Confidence Score    : {confidence:.2f}%\n")

In [ ]:
### voice chatbot

In [ ]:
!pip install -q elevenlabs openai-whisper groq

In [ ]:
import os
from getpass import getpass

# Enter your API keys
os.environ["ELEVENLABS_API_KEY"] = getpass("Enter ElevenLabs API Key: ")
os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

In [ ]:
from google.colab import output
from IPython.display import HTML, Javascript, display
import base64

# JavaScript snippet to record audio in the browser
JS_AUDIO_RECORDER = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});

var recordAudio = async function(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const mediaRecorder = new MediaRecorder(stream);
  const audioChunks = [];

  mediaRecorder.addEventListener("dataavailable", event => {
    audioChunks.push(event.data);
  });

  mediaRecorder.start();
  console.log("Recording started...");
  await sleep(seconds * 1000);

  const stopped = new Promise((resolve, reject) => {
    mediaRecorder.onstop = resolve;
    mediaRecorder.onerror = reject;
  });

  mediaRecorder.stop();
  await stopped;
  stream.getTracks().forEach(track => track.stop());
  console.log("Recording stopped.");

  const audioBlob = new Blob(audioChunks);
  const base64Data = await b2text(audioBlob);
  return base64Data;
};
"""

def record_audio(filename="input.wav", duration=5):
    """Triggers microphone recording in browser and saves it locally."""
    print(f"🎤 Recording for {duration} seconds... Please speak into your mic!")
    display(Javascript(JS_AUDIO_RECORDER))
    b64_audio = output.eval_js(f"recordAudio({duration})")

    # Strip base64 metadata header and save binary audio
    audio_bytes = base64.b64decode(b64_audio.split(",")[1])
    with open(filename, "wb") as f:
        f.write(audio_bytes)
    print("✅ Recording finished and saved!")

In [ ]:
import whisper
from groq import Groq
from elevenlabs.client import ElevenLabs
from IPython.display import Audio, display

# 1. Load Whisper for local STT
print("Loading Whisper STT model...")
stt_model = whisper.load_model("base")

# 2. Initialize LLM Client (Groq for ultra-low latency)
llm_client = Groq(api_key=os.environ["GROQ_API_KEY"])

# 3. Initialize ElevenLabs TTS Client
eleven_client = ElevenLabs(api_key=os.environ["ELEVENLABS_API_KEY"])

# Conversation memory history
conversation_history = [
    {
        "role": "system",
        "content": "You are a concise, helpful voice assistant. Keep answers brief, natural, and conversational without markdown formatting."
    }
]

Loading Whisper STT model...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 170MiB/s]


In [ ]:
def speech_to_text(audio_file_path="input.wav"):
    """Transcribes user speech to text."""
    result = stt_model.transcribe(audio_file_path)
    return result["text"].strip()

def query_llm(user_prompt):
    """Sends transcription to LLM and returns the text response."""
    conversation_history.append({"role": "user", "content": user_prompt})

    completion = llm_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=conversation_history,
        temperature=0.7,
        max_tokens=250
    )

    bot_reply = completion.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": bot_reply})
    return bot_reply

def text_to_speech(text, output_file="response.mp3"):
    """Converts LLM text to speech using ElevenLabs and plays audio in Colab."""
    # Common Default Voices: 'JBFqnCBsd6RMkjVDRZzb' (George), '21m00Tcm4TlvDq8ikWAM' (Rachel)
    audio_stream = eleven_client.text_to_speech.convert(
        text=text,
        voice_id="JBFqnCBsd6RMkjVDRZzb",
        model_id="eleven_multilingual_v2",
        output_format="mp3_44100_128"
    )

    # Save audio stream to MP3
    with open(output_file, "wb") as f:
        for chunk in audio_stream:
            f.write(chunk)

    # Auto-play the response in the notebook
    display(Audio(output_file, autoplay=True))

In [ ]:
def run_voice_turn(duration=5):
    # 1. Record User Voice
    record_audio(filename="input.wav", duration=duration)

    # 2. STT: Transcribe Audio
    user_text = speech_to_text("input.wav")
    print(f"\n👤 You said: {user_text}")

    if not user_text:
        print("⚠️ No speech detected. Please try again.")
        return

    # 3. LLM: Generate Answer
    bot_text = query_llm(user_text)
    print(f"🤖 Bot: {bot_text}\n")

    # 4. TTS: Synthesize and Play
    print("🔊 Synthesizing voice with ElevenLabs...")
    text_to_speech(bot_text)

# Run a 6-second recording session
run_voice_turn(duration=6)

🎤 Recording for 6 seconds... Please speak into your mic!


<IPython.core.display.Javascript object>

✅ Recording finished and saved!


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



👤 You said: Tell me about the noises how I will arrive in the...
🤖 Bot: Sure—just let me know where you’re headed, and I can paint the sounds you’ll hear on the way.

🔊 Synthesizing voice with ElevenLabs...
